# 01 — Vollständiger Transfer: Global-6 → Sensor 7

Das Notebook startet den vollständigen Workflow aus `calibration_transfer_sensor0.py`. Von jedem der sieben Geräte wird ausschließlich Subsensor 0 verwendet. Die sechs Quellgeräte dienen abwechselnd als LOSO-Pseudoziel, damit die Hyperparameter aller Transfermethoden ohne Zugriff auf Sensor 7 gewählt werden. Verglichen werden Global-6, Global+Target, Scratch, Head-only, Fine-tune, DS und PDS.

## Versuchslogik

1. Sechs LOSO-Folds: fünf Quellgeräte trainieren das Fold-Globalmodell, das sechste ist Pseudoziel.
2. Für jedes Budget werden Lernrate/Epochen von Global+Target, Scratch, Head-only und Fine-tune sowie Alpha/Fenster von DS/PDS per mittlerem LOSO-UGM-RMSE gewählt.
3. Erst nach dieser Auswahl wird das finale Global-6-Modell trainiert.
4. Die eingefrorenen Einstellungen werden auf Sensor 7 angewandt. Sensor 7 beeinflusst die Auswahl nicht.
5. Harte Formprüfung: Jeder Zyklus hat `(1, 1440, 1)`; Sensoren werden niemals als Eingangskanäle gestapelt.

In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import Image, display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'Networks' / 'TCOCNNv3.py').exists())
SCRIPT = ROOT / 'Evaluation Seminar' / 'Day_04' / 'calibration_transfer_sensor0.py'
OUTPUT = ROOT / 'artifacts' / 'seminar_day4_global6_sensor0'
GLOBAL_PARAMS = ROOT / 'artifacts' / 'seminar_day4_global6_hpo' / 'best_hyperparameters.json'
if not GLOBAL_PARAMS.exists():
    raise FileNotFoundError('Zuerst 00_Global6_Hyperparameter_Optimization.py ausführen: ' + str(GLOBAL_PARAMS))

# Alle tatsächlich getesteten Hyperparameter stehen hier sichtbar.
BUDGETS = [5, 10, 20, 40, 80, 120, 137]
SOURCE_EPOCHS = 100
CANDIDATE_EPOCHS = [10, 20, 40, 60]
LR_GLOBAL_TARGET = [1e-4, 3e-4, 1e-3]
LR_SCRATCH = [1e-4, 3e-4, 1e-3]
LR_HEAD_ONLY = [1e-5, 3e-5, 1e-4, 3e-4]
LR_FINE_TUNE = [1e-6, 3e-6, 1e-5, 3e-5, 1e-4]
DS_PDS_ALPHAS = [.001, .01, .1, 1., 10., 1000.]
PDS_WINDOWS = [5, 9, 21]
LOSO_FOLDS = 6

def values(option, items):
    return [option, *map(str, items)]

command = ([sys.executable, '-u', str(SCRIPT), '--output', str(OUTPUT),
            '--global-params', str(GLOBAL_PARAMS),
            '--source-epochs', str(SOURCE_EPOCHS), '--loso-folds', str(LOSO_FOLDS)]
           + values('--budgets', BUDGETS)
           + values('--candidate-epochs', CANDIDATE_EPOCHS)
           + values('--lr-global-target', LR_GLOBAL_TARGET)
           + values('--lr-scratch', LR_SCRATCH)
           + values('--lr-head', LR_HEAD_ONLY)
           + values('--lr-finetune', LR_FINE_TUNE)
           + values('--alphas', DS_PDS_ALPHAS)
           + values('--windows', PDS_WINDOWS))
print('LOSO-Folds:', LOSO_FOLDS)
print('Budgets:', BUDGETS)
print('Epochenkandidaten:', CANDIDATE_EPOCHS)
print('LR Global+Target:', LR_GLOBAL_TARGET)
print('LR Scratch:', LR_SCRATCH)
print('LR Head-only:', LR_HEAD_ONLY)
print('LR Fine-tune:', LR_FINE_TUNE)
print('DS/PDS Alpha:', DS_PDS_ALPHAS, '; PDS-Fenster:', PDS_WINDOWS)

## LOSO-Suche und finaler Transfer

Der vollständige Lauf ist rechenintensiv, weil auch die neuronalen Transfermethoden in jedem LOSO-Fold neu trainiert werden. Die Ausgabe zeigt zuerst sämtliche Entwicklungsfolds und danach die finale Anwendung auf Sensor 7.

In [ ]:
subprocess.run(command, cwd=ROOT, check=True)

## Audit der Auswahl

Die folgenden Dateien dokumentieren die sechs LOSO-Folds, die daraus gewählten Einstellungen und bestätigen, dass Sensor 7 nicht zur Auswahl verwendet wurde.

In [ ]:
config = json.loads((OUTPUT / 'run_config.json').read_text(encoding='utf-8'))
selected = json.loads((OUTPUT / 'selected_loso_settings.json').read_text(encoding='utf-8'))
print(json.dumps(config, indent=2))
print(f'Gewählte Kombinationen: {len(selected)}')
try:
    import pandas as pd
    selected_table = pd.DataFrame(selected).sort_values(['budget_UGMs', 'method'])
    display(selected_table[['budget_UGMs', 'method', 'lr', 'epoch', 'alpha', 'radius',
                            'development_RMSE']])
except ImportError:
    print(json.dumps(selected, indent=2))

## Lernkurven und Resultate

Gezeigt werden die Train/Validierungs-Lernkurve des finalen Global-6-Modells, die LOSO-Auswahlkurven und der Vergleich aller Methoden auf Sensor 7.

In [ ]:
for filename in ['source_training_curve.png', 'loso_selected_settings.png',
                 'all_methods_budget_curve.png']:
    print(filename)
    display(Image(filename=str(OUTPUT / filename)))

In [ ]:
metrics_rows = json.loads((OUTPUT / 'final_metrics.json').read_text(encoding='utf-8'))
try:
    import pandas as pd
    display(pd.DataFrame(metrics_rows))
except ImportError:
    print(json.dumps(metrics_rows, indent=2))